In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing import image
from PIL import ImageFile

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

ImageFile.LOAD_TRUNCATED_IMAGES = True


In [2]:
# STEP 1 — LOAD DATA
data_dir = "../../data sheets/training_set"
batch_size = 32
img_size = (224, 224)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.25,
    height_shift_range=0.25,
    zoom_range=0.3,
    horizontal_flip=True,
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset="training",
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset="validation",
    shuffle=False
)

Found 4000 images belonging to 5 classes.
Found 1000 images belonging to 5 classes.


In [3]:
# STEP 2 — FINE-TUNE RESNET50

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))

for layer in base_model.layers[:-20]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation="relu")(x)
x = Dropout(0.4)(x)
outputs = Dense(5, activation="softmax")(x)

ft_model = Model(inputs=base_model.input, outputs=outputs)

ft_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)

ft_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[early_stop]
)


Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 639s 5s/step - accuracy: 0.3322 - loss: 1.5902 - val_accuracy: 0.4170 - val_loss: 1.3801
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 499s 4s/step - accuracy: 0.4568 - loss: 1.3173 - val_accuracy: 0.4710 - val_loss: 1.2977
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 512s 4s/step - accuracy: 0.5337 - loss: 1.1764 - val_accuracy: 0.4870 - val_loss: 1.2561
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 478s 4s/step - accuracy: 0.5830 - loss: 1.0613 - val_accuracy: 0.5340 - val_loss: 1.1989
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 515s 4s/step - accuracy: 0.6315 - loss: 0.9606 - val_accuracy: 0.5190 - val_loss: 1.2339


In [4]:
# STEP 3 — FEATURE EXTRACTION
feature_extractor = Model(
    inputs=ft_model.input,
    outputs=ft_model.get_layer("conv5_block3_out").output
)

def extract_features(generator):
    features, labels = [], []
    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        feats = feature_extractor.predict(x_batch, verbose=0)
        feats = feats.reshape(feats.shape[0], -1)
        features.append(feats)
        labels.append(y_batch)
        if (i+1)*batch_size >= generator.n:
            break
    return np.vstack(features), np.hstack(labels)

X_train, y_train = extract_features(train_gen)
X_test, y_test = extract_features(val_gen)

In [5]:
# STEP 4 — SCALING + PCA

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

pca = PCA(n_components=256, random_state=42)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)


In [6]:
# STEP 5 — LOGISTIC REGRESSION (TRAINING)

clf = LogisticRegression(
    max_iter=2000,
    multi_class="multinomial",
    solver="lbfgs",
    n_jobs=-1
)

clf.fit(X_train, y_train)

c:\Users\themi\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,2000
,multi_class,'multinomial'


In [7]:
# STEP 6 — EVALUATION

y_pred = clf.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred)*100, 2), "%")
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))



Accuracy: 50.9 %

Classification Report:
               precision    recall  f1-score   support

         0.0       0.50      0.57      0.54       200
         1.0       0.55      0.57      0.56       200
         2.0       0.37      0.36      0.37       200
         3.0       0.61      0.55      0.58       200
         4.0       0.52      0.49      0.51       200

    accuracy                           0.51      1000
   macro avg       0.51      0.51      0.51      1000
weighted avg       0.51      0.51      0.51      1000


Confusion Matrix:
 [[115  32  27   7  19]
 [ 34 114  23   9  20]
 [ 50  22  72  25  31]
 [ 14  18  36 109  23]
 [ 16  22  35  28  99]]


In [8]:

# STEP 7 — SAVE MODELS
os.makedirs("models", exist_ok=True)

ft_model.save("models/fine_tuned_resnet50_LR.keras")
joblib.dump(scaler, "models/scaler_LR.pkl")
joblib.dump(pca, "models/pca_LR.pkl")
joblib.dump(clf, "models/logistic_regression_face_shape.pkl")


['models/logistic_regression_face_shape.pkl']

In [9]:
# STEP 8 — HAIRSTYLE RECOMMENDER

def recommend_hairstyle(face_shape):
    data = {
        "Heart": ["Side-swept bangs", "Chin-length bobs", "Soft layers"],
        "Oblong": ["Curtain bangs", "Wavy volume", "Shoulder cuts"],
        "Oval": ["Any style", "Long waves", "Pixie cut"],
        "Round": ["Long layers", "Side parts", "High ponytail"],
        "Square": ["Soft curls", "Layered cuts", "Side fringes"]
    }
    return data.get(face_shape, ["Consult stylist"])


In [10]:
# STEP 9 — PREDICT UNSEEN IMAGE

def predict_face_shape(img_path):
    img = image.load_img(img_path, target_size=(224,224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    feats = feature_extractor.predict(img_array, verbose=0)
    feats = feats.reshape(1, -1)

    feats = scaler.transform(feats)
    feats = pca.transform(feats)

    pred = clf.predict(feats)[0]
    labels = ['Heart', 'Oblong', 'Oval', 'Round', 'Square']
    return labels[int(pred)]

In [11]:
test_images = [
    "../../data sheets/kirula.png",
    "../../data sheets/heshan.jpg",
    "../../data sheets/dewni.jpg",
    "../../data sheets/1.jpg"
]

for img in test_images:
    if os.path.exists(img):
        print(img, "→", predict_face_shape(img))
    else:
        print(img, "not found")


../../data sheets/kirula.png → Oblong
../../data sheets/heshan.jpg → Oblong
../../data sheets/dewni.jpg → Square
../../data sheets/1.jpg → Oblong


In [12]:


test_img = "../../data sheets/zendaya.jpg"

if os.path.exists(test_img):
    shape = predict_face_shape(test_img)
    print("\nPredicted Face Shape:", shape)
    print("Recommended Hairstyles:")
    for h in recommend_hairstyle(shape):
        print("-", h)
else:
    print("Test image not found")


Predicted Face Shape: Heart
Recommended Hairstyles:
- Side-swept bangs
- Chin-length bobs
- Soft layers


In [13]:
test_img = "../../data sheets/z.jpg"

if os.path.exists(test_img):
    shape = predict_face_shape(test_img)
    print("\nPredicted Face Shape:", shape)
    print("Recommended Hairstyles:")
    for h in recommend_hairstyle(shape):
        print("-", h)
else:
    print("Test image not found")


Predicted Face Shape: Heart
Recommended Hairstyles:
- Side-swept bangs
- Chin-length bobs
- Soft layers
